# Break-even price and power purchase agreement
HyDesign calculates the price of electricity and/or hydrogen that would break-even i.e. result in NPV of zero. These results are based on the current hybrid configuration and operation. This means that the results will vary depending on which input parameters you are using, but will generally converge when the input price parameters are approaching the resulting break-even prices.

## Break-even price of H2
The break-even price of H2 can be obtained as following.

In [ ]:
# Install hydesign if needed
import importlib
if not importlib.util.find_spec("hydesign"):
    !pip install git+https://gitlab.windenergy.dtu.dk/TOPFARM/hydesign.git   

In [ ]:
%%capture
import pandas as pd
from hydesign.assembly.hpp_assembly import hpp_model
from hydesign.assembly.hpp_assembly_P2X import hpp_model_P2X
from hydesign.examples import examples_filepath

In [ ]:
examples_sites = pd.read_csv(f'{examples_filepath}examples_sites.csv', index_col=0, sep=';')
name = 'France_good_wind'
ex_site = examples_sites.loc[examples_sites.name == name]
longitude = ex_site['longitude'].values[0]
latitude = ex_site['latitude'].values[0]
altitude = ex_site['altitude'].values[0]
input_ts_fn = examples_filepath+ex_site['input_ts_fn'].values[0]
sim_pars_fn = examples_filepath+ex_site['sim_pars_fn'].values[0]
H2_demand_fn = examples_filepath+ex_site['H2_demand_col'].values[0]
hpp = hpp_model_P2X(
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    num_batteries = 1,
    work_dir = './',
    sim_pars_fn = sim_pars_fn,
    input_ts_fn = input_ts_fn,
    H2_demand_fn=H2_demand_fn) 


In [ ]:
x = [10.0, 350.0, 5.0, 20.0, 7.0, 100.0, 50.0, 180.0, 1.5, 0.0, 3.0, 5.0, 100.0, 2500.0]
outs = hpp.evaluate(*x)

In [ ]:
hpp.print_design()

The break-even H2 price is seen in the above table and can also be extracted directly:

In [ ]:
print(f'Input H2 price: {hpp.sim_pars["price_H2"]:.2f}')
print(f'Break-even H2 price: {float(hpp.prob["break_even_H2_price"][0]):.2f}')

## Power Purchase Agreement (PPA)
Constant price can be introduced by supplying a time series of constant prices or setting the PPA price directly when instantiating the hpp model:

In [ ]:
PPA = 40 # Euro/MWh
hpp = hpp_model(
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    num_batteries = 1,
    work_dir = './',
    sim_pars_fn = sim_pars_fn,
    input_ts_fn = input_ts_fn,
    ppa_price=PPA,)

In [ ]:
x = [10.0, 350.0, 5.0, 20.0, 7.0, 100.0, 50.0, 180.0, 1.5, 0.0, 3.0, 5.0]
outs = hpp.evaluate(*x)

In [ ]:
hpp.print_design()

The break-even PPA price is seen in the above table and can also be extracted directly:

In [ ]:
print(f'Input H2 price: {float(PPA):.2f}')
print(f'Break-even H2 price: {float(hpp.prob["break_even_PPA_price"][0]):.2f}')

LCOE is a measure of lifetime cost of producing power, but comes out somewhat different from the break-even price. This is because break-even price is based on NPV calculations that takes into account e.g. the corporate tax rate as well as penalties incurred from not meeting expected loads.